In [0]:
# COMMAND ----------
from pyspark.sql.functions import current_timestamp, lit

# 1. Configuration - Aligned with your healthcare_data.py output
raw_path = "/Volumes/workspace/healthcare2/healthcare_db/healthcare_raw.csv"
bronze_table = "workspace.healthcare3.bronze_patients"

# 2. Optimized Reading
# We use header=true and inferSchema=true to capture the specific types 
# (integers for wait times, doubles for billing) from your professional script.
df_raw = (spark.read
          .format("csv")
          .option("header", "true")
          .option("inferSchema", "true")
          .load(raw_path))

# 3. Audit Metadata & Data Lineage (High Impact for Senior Roles)
# Replace input_file_name with _metadata.file_path for Unity Catalog compatibility.
df_bronze = (df_raw
             .withColumn("ingestion_timestamp", current_timestamp())
             .withColumn("source_file", df_raw["_metadata.file_path"])
             .withColumn("data_load_version", lit("1.0")))

# 4. Write to Delta Table
# Using 'mergeSchema' allows for future-proofing if the CMO adds new clinical metrics.
(df_bronze.write
 .format("delta")
 .mode("overwrite")
 .option("mergeSchema", "true") 
 .saveAsTable(bronze_table))

# 5. Small Output for Validation
print(f"✅ Success: Bronze layer loaded into {bronze_table}")
display(df_bronze.limit(5))

✅ Success: Bronze layer loaded into workspace.healthcare3.bronze_patients


patient_id,encounter_id,age,gender,blood_group,admission_date,discharge_date,treatment_type,blood_pressure_status,heart_rate,oxygen_saturation,icd10_history,lab_result_status,dept_name,wait_time_min,bed_occupancy_status,is_emergency_admission,staff_ratio,payer_name,claim_status,denial_reason,total_billed_amount,claim_amount,approved_claim_amount,ingestion_timestamp,source_file,data_load_version
PAT-0000001,ENC-0000001,1,Male,AB-,2024-11-06,2024-11-14,Medical,Normal,55,93,"I25,J45",Pending,General Med,117,Available,0,2026-03-20T01:04:00.000Z,Medicare,Paid,None,11106.48,10123.17,9976.56,2026-03-20T20:20:51.746Z,dbfs:/Volumes/workspace/healthcare2/healthcare_db/healthcare_raw.csv,1.0
PAT-0000002,ENC-0000002,45,Male,B-,2024-04-12,2024-04-13,Emergency,Normal,88,91,I25,Abnormal,ER,118,Cleaning,1,2026-03-20T01:06:00.000Z,BlueCross,Denied,Coding_Error,19919.22,18839.71,3618.73,2026-03-20T20:20:51.746Z,dbfs:/Volumes/workspace/healthcare2/healthcare_db/healthcare_raw.csv,1.0
PAT-0000003,ENC-0000003,82,Male,O-,2024-06-23,2024-06-29,Diagnostic,Normal,118,95,I25,Abnormal,Rehab,166,Occupied,0,2026-03-20T01:06:00.000Z,Self-Pay,Denied,Coding_Error,15165.51,14407.11,4405.4,2026-03-20T20:20:51.746Z,dbfs:/Volumes/workspace/healthcare2/healthcare_db/healthcare_raw.csv,1.0
PAT-0000004,ENC-0000004,35,Male,AB-,2024-07-27,2024-08-01,Medical,Normal,94,99,"J45,I10",Critical,General Med,93,Occupied,1,2026-03-20T01:06:00.000Z,Self-Pay,Denied,Prior_Auth,14159.42,13585.74,4238.47,2026-03-20T20:20:51.746Z,dbfs:/Volumes/workspace/healthcare2/healthcare_db/healthcare_raw.csv,1.0
PAT-0000005,ENC-0000005,10,Other,AB-,2025-11-11,2025-11-29,Surgical,Normal,120,82,"J45,J45,I25",Normal,ICU,374,Occupied,1,2026-03-20T01:02:00.000Z,BlueCross,Paid,None,41363.97,37674.43,37495.96,2026-03-20T20:20:51.746Z,dbfs:/Volumes/workspace/healthcare2/healthcare_db/healthcare_raw.csv,1.0
